In [1]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict
from dotenv import load_dotenv

d:\dev\Tutorials\LangGraph\myenv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [2]:
load_dotenv()

True

In [3]:
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash")


In [4]:
# Create state
class LLMState(TypedDict) :
    question: str
    answer: str

In [5]:
def llm_qa(state: LLMState)-> LLMState:
    # Extract the question from state
    question  =  state['question']

    # form a prompt
    prompt = f'Answer the following question {question}'

    # Ask that question form model
    answer = model.invoke(prompt).content

    #update answer in the state
    state['answer'] = answer

    return state


In [6]:
# Create graph
graph = StateGraph(LLMState)
#add nodes
graph.add_node('llm_qa', llm_qa)

#add edges
graph.add_edge(START,'llm_qa')
graph.add_edge('llm_qa',END)

# compile the graph
workflow = graph.compile()


In [7]:
initial_state = {'question': 'How far is Sun from Earth? '}
final_state = workflow.invoke(initial_state)
print(final_state)

{'question': 'How far is Sun from Earth? ', 'answer': "The distance between the Sun and Earth is not constant, as Earth travels in an elliptical orbit around the Sun. However, we can provide an average distance and the range:\n\n*   **Average Distance:** Approximately **150 million kilometers (km)** or **93 million miles**.\n*   **Astronomical Unit (AU):** This average distance is defined as **1 Astronomical Unit (AU)**, which is a standard unit of measurement in astronomy.\n\n**Due to Earth's elliptical orbit:**\n\n*   **Perihelion (closest point):** Around **147.1 million km (91.4 million miles)**. This usually occurs in early January.\n*   **Aphelion (farthest point):** Around **152.1 million km (94.5 million miles)**. This usually occurs in early July.\n\nSo, the most common and useful answer is **150 million kilometers (93 million miles)**, or simply **1 AU**."}
